# İstanbul Toplu Ulaşım (İETT) Saatlik Yolcu Analizi

**Veri Bilimi Dönem Projesi**

İş akışı: veri yükleme → tanıma → temizleme → EDA (4 görselleştirme) →
3 araştırma sorusu → modelleme (baseline + RandomForest, zamansal ayrım) → yorum.

**Veri kaynağı:** İBB Açık Veri Portalı — Saatlik Toplu Ulaşım Veri Seti (BELBİM A.Ş.)
**Lisans:** İstanbul Büyükşehir Belediyesi Açık Veri Lisansı


## 0. Kütüphaneler ve görsel tema

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Görsel tema: modern, canlı seaborn paleti ---
sns.set_theme(style="whitegrid", context="notebook")
PALET = sns.color_palette("Set2")           # canlı, ayırt edilebilir renkler
sns.set_palette(PALET)
plt.rcParams["figure.figsize"]  = (10, 5)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.titlesize"]   = 13
ANA_RENK = PALET[2]                          # tek seri grafiklerde kullanılacak vurgu


## 1. Sütun Eşleme (tek ayar noktası)

> ÖNEMLİ: Önce Adım 2'yi çalıştırıp `df.columns` çıktısını gör. Aşağıdaki sözlükteki
> sağ taraftaki değerleri GERÇEK sütun adlarıyla değiştir. Notebook'un geri kalanı
> bu eşlemeyi kullanır; başka hiçbir yeri elle değiştirmene gerek kalmaz.


In [ ]:
# Sol taraf: kodun kullandığı standart adlar — DEĞİŞTİRME
# Sağ taraf: CSV'deki gerçek sütun adları — Adım 2'den sonra DÜZELT
S = {
    "tarih":        "transition_date",        # tarih-saat sütunu
    "tur":          "transport_type_desc",    # ulaşım türü (OTOBÜS, METRO, ...)
    "hat":          "line",                   # hat adı
    "yolcu":        "number_of_passenger",    # yolcu sayısı (hedef değişken)
}
OTOBUS_DEGERI = "OTOBÜS"   # 'tur' sütununda otobüsü ifade eden değer


## 2. Veri Yükleme ve Tanıma

In [ ]:
df = pd.read_csv("../data/raw/iett_2023_raw.csv")
df.head()


In [ ]:
df.info()


In [ ]:
print("Sütunlar:", list(df.columns))
df.isna().sum()


In [ ]:
df.describe()


## 3. Temizleme

Adımlar: tarih dönüşümü → türetilmiş zaman sütunları → tatil işaretleme →
otobüse filtreleme → eksik/aykırı değer → kaydetme.


In [ ]:
# --- 3.1 Tarih dönüşümü ve türetilmiş zaman sütunları ---
df["tarih"] = pd.to_datetime(df[S["tarih"]])
df["saat"]          = df["tarih"].dt.hour
df["gun"]           = df["tarih"].dt.day_name()
df["haftanin_gunu"] = df["tarih"].dt.dayofweek      # 0=Pzt ... 6=Paz
df["hafta_sonu_mu"] = df["haftanin_gunu"].isin([5, 6])
df["ay"]            = df["tarih"].dt.month
df["tarih_gun"]     = df["tarih"].dt.normalize()    # tatil eşlemesi için
df[["tarih", "saat", "gun", "hafta_sonu_mu", "ay"]].head()


In [ ]:
# --- 3.2 Resmî tatil işaretleme (2023 Türkiye resmî tatilleri) ---
RESMI_TATILLER_2023 = pd.to_datetime([
    "2023-01-01",                                   # Yılbaşı
    "2023-04-21", "2023-04-22", "2023-04-23",       # Ramazan Bayramı
    "2023-04-23",                                   # Ulusal Egemenlik ve Çocuk Bayramı
    "2023-05-01",                                   # Emek ve Dayanışma Günü
    "2023-05-19",                                   # Gençlik ve Spor Bayramı
    "2023-06-28", "2023-06-29", "2023-06-30", "2023-07-01",  # Kurban Bayramı
    "2023-07-15",                                   # Demokrasi ve Millî Birlik Günü
    "2023-08-30",                                   # Zafer Bayramı
    "2023-10-29",                                   # Cumhuriyet Bayramı
])
df["tatil_mi"] = df["tarih_gun"].isin(RESMI_TATILLER_2023)
print("Tatil günü kayıt sayısı:", int(df["tatil_mi"].sum()))


In [ ]:
# --- 3.3 Otobüse filtrele (tema: İETT) ---
otobus = df[df[S["tur"]].astype(str).str.upper() == OTOBUS_DEGERI].copy()
print("Otobüs satır sayısı:", len(otobus))


In [ ]:
# --- 3.4 Eksik ve aykırı değerler ---
y = S["yolcu"]
otobus[y] = pd.to_numeric(otobus[y], errors="coerce")   # tip güvenliği
print("Eksik yolcu değeri:", otobus[y].isna().sum())
otobus = otobus.dropna(subset=[y])                      # az sayıda ise sil

q1, q3 = otobus[y].quantile([0.25, 0.75])
ust = q3 + 1.5 * (q3 - q1)                              # IQR üst sınırı
print(f"IQR üst aykırı sınırı: {ust:.0f}  |  Sınır üstü kayıt: {(otobus[y] > ust).sum()}")
# Not: Aykırıları silmek yerine işaretliyoruz; gerçek pik saatleri silmemek için.
otobus["aykiri_mi"] = otobus[y] > ust


In [ ]:
# --- 3.5 Temiz veriyi kaydet ---
otobus.to_csv("../data/processed/iett_temiz.csv", index=False, encoding="utf-8")
print("Temiz veri kaydedildi:", otobus.shape)


## 4. Keşifsel Veri Analizi (4 görselleştirme)

### 4.1 Çizgi — Saate göre ortalama yolcu

In [ ]:
saatlik = otobus.groupby("saat")[S["yolcu"]].mean()
ax = saatlik.plot(marker="o", color=ANA_RENK, linewidth=2)
ax.set(title="Saate Göre Ortalama Yolcu Sayısı", xlabel="Saat", ylabel="Ortalama yolcu")
plt.tight_layout(); plt.savefig("../figures/01_saatlik_cizgi.png", dpi=120, bbox_inches="tight")
plt.show()


### 4.2 Histogram — Yolcu sayısı dağılımı

In [ ]:
ax = otobus[S["yolcu"]].plot(kind="hist", bins=50, color=PALET[1], edgecolor="white")
ax.set(title="Yolcu Sayısı Dağılımı", xlabel="Yolcu sayısı")
plt.tight_layout(); plt.savefig("../figures/02_histogram.png", dpi=120, bbox_inches="tight")
plt.show()


### 4.3 Kutu — Aylara göre yolcu (mevsim temsili)

In [ ]:
ax = sns.boxplot(data=otobus, x="ay", y=S["yolcu"])
ax.set(title="Aylara Göre Yolcu Sayısı Dağılımı", xlabel="Ay", ylabel="Yolcu sayısı")
plt.tight_layout(); plt.savefig("../figures/03_kutu_ay.png", dpi=120, bbox_inches="tight")
plt.show()


### 4.4 Isı haritası — Saat × haftanın günü

In [ ]:
pivot = otobus.pivot_table(index="haftanin_gunu", columns="saat",
                           values=S["yolcu"], aggfunc="mean")
ax = sns.heatmap(pivot, cmap="rocket_r")
ax.set(title="Ortalama Yolcu: Saat × Haftanın Günü", xlabel="Saat", ylabel="Gün (0=Pzt)")
plt.tight_layout(); plt.savefig("../figures/04_isi_haritasi.png", dpi=120, bbox_inches="tight")
plt.show()


## 5. Araştırma Soruları

### Soru 1 — Gün-içi pik saatler; hafta içi vs hafta sonu

In [ ]:
hi = otobus[~otobus["hafta_sonu_mu"]].groupby("saat")[S["yolcu"]].mean()
hs = otobus[otobus["hafta_sonu_mu"]].groupby("saat")[S["yolcu"]].mean()
ax = hi.plot(marker="o", label="Hafta içi", linewidth=2)
hs.plot(marker="s", label="Hafta sonu", linewidth=2, ax=ax)
ax.set(title="Hafta İçi vs Hafta Sonu Saatlik Profil", xlabel="Saat", ylabel="Ortalama yolcu")
ax.legend(); plt.tight_layout()
plt.savefig("../figures/05_haftaici_haftasonu.png", dpi=120, bbox_inches="tight"); plt.show()


**Bulgu (kendi grafiğinle teyit et):** Hafta içi sabah (~08:00) ve akşam (~17–18:00)
çift tepe görülür; bu işe gidiş-dönüş örüntüsüyle uyumludur. Hafta sonu bu tepeler
silinir ve talep güne daha düz yayılır.

### Soru 2 — Mevsimsellik var mı?

In [ ]:
aylik = otobus.groupby("ay")[S["yolcu"]].sum()
ax = aylik.plot(kind="bar", color=PALET[:len(aylik)])
ax.set(title="Aylara Göre Toplam Yolcu", xlabel="Ay", ylabel="Toplam yolcu")
plt.tight_layout(); plt.savefig("../figures/06_aylik_bar.png", dpi=120, bbox_inches="tight")
plt.show()


**Bulgu (teyit et):** Temmuz (yaz / okul tatili) ayında toplam yolcu, diğer aylara
göre belirgin düşer. Bu, eğitim takviminin toplu taşıma talebini sürüklediğini gösterir.

### Soru 3 — Az sayıda hat çok mu yolcu taşıyor? (Pareto)

In [ ]:
hat = otobus.groupby(S["hat"])[S["yolcu"]].sum().sort_values(ascending=False)
kum = hat.cumsum() / hat.sum()
ilk20_pay = kum.iloc[max(int(len(hat) * 0.2) - 1, 0)] * 100
print("En yoğun 10 hat:\n", hat.head(10))
print(f"\nİlk %20 hattın toplam yolcudaki payı: %{ilk20_pay:.1f}")

ax = (kum.reset_index(drop=True) * 100).plot(color=ANA_RENK, linewidth=2)
ax.axvline(len(hat) * 0.2, color="gray", linestyle="--")
ax.set(title="Hatların Kümülatif Yolcu Payı (Pareto)",
       xlabel="Hat sırası (yoğundan seyreğe)", ylabel="Kümülatif yolcu payı (%)")
plt.tight_layout(); plt.savefig("../figures/07_pareto.png", dpi=120, bbox_inches="tight")
plt.show()


**Bulgu (teyit et):** Hatların ilk %20'si toplam yolcunun büyük çoğunluğunu taşır
(80-20 / Pareto etkisi). Yani talep az sayıda yoğun koridorda toplanmıştır.

## 6. Modelleme

Hedef: saatlik toplam yolcuyu zaman özelliklerinden tahmin etmek. İki kritik nokta:
- **Zamansal ayrım:** Bu bir zaman serisi olduğu için rastgele bölme gelecekten geçmişe
  bilgi sızdırır (data leakage). Bu yüzden ilk aylarla eğitip son ayla test ediyoruz.
- **Baseline:** Modelin gerçekten değer kattığını göstermek için naif bir tahminciyle
  (her saatin ortalaması) karşılaştırıyoruz.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Saat-gün-ay-tatil bazında toplam yolcu
model_df = (otobus
            .groupby(["ay", "haftanin_gunu", "saat", "hafta_sonu_mu", "tatil_mi"])[S["yolcu"]]
            .sum().reset_index())

ozellikler = ["ay", "haftanin_gunu", "saat", "hafta_sonu_mu", "tatil_mi"]
X = model_df[ozellikler].astype(int)
y_hedef = model_df[S["yolcu"]]


In [ ]:
# --- Zamansal ayrım: son ay (en büyük 'ay' değeri) test, gerisi eğitim ---
test_ay = model_df["ay"].max()
egitim = model_df["ay"] < test_ay
X_train, X_test = X[egitim], X[~egitim]
y_train, y_test = y_hedef[egitim], y_hedef[~egitim]
print(f"Eğitim: {len(X_train)} satır  |  Test (ay={test_ay}): {len(X_test)} satır")


In [ ]:
# --- Baseline: her saatin eğitim ortalaması ---
saat_ort = y_train.groupby(X_train["saat"]).mean()
baseline_pred = X_test["saat"].map(saat_ort).fillna(y_train.mean())
baseline_mae = mean_absolute_error(y_test, baseline_pred)
print(f"Baseline MAE: {baseline_mae:,.0f}")


In [ ]:
# --- RandomForest ---
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

rf_mae = mean_absolute_error(y_test, pred)
rf_r2  = r2_score(y_test, pred)
iyilesme = (baseline_mae - rf_mae) / baseline_mae * 100
print(f"RandomForest  R²: {rf_r2:.3f}  |  MAE: {rf_mae:,.0f}")
print(f"Baseline'a göre MAE iyileşmesi: %{iyilesme:.1f}")


In [ ]:
# --- Gerçek vs tahmin ---
plt.scatter(y_test, pred, alpha=0.6, color=ANA_RENK, edgecolor="white")
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, "--", color="gray")
plt.xlabel("Gerçek"); plt.ylabel("Tahmin"); plt.title("Gerçek vs Tahmin (test ayı)")
plt.tight_layout(); plt.savefig("../figures/08_gercek_vs_tahmin.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# --- Özellik önemleri ---
onem = pd.Series(model.feature_importances_, index=ozellikler).sort_values()
ax = onem.plot(kind="barh", color=PALET[3])
ax.set(title="Özellik Önemleri", xlabel="Önem")
plt.tight_layout(); plt.savefig("../figures/09_ozellik_onem.png", dpi=120, bbox_inches="tight")
plt.show()


**Bulgu (teyit et):** RandomForest, baseline'ı %X iyileştirdi; demek ki saatin yanı
sıra gün/ay/tatil bilgisi de talebi açıklıyor. Özellik önemlerinde en güçlü değişken
genelde *saat*tir — talebin en çok günün saatine bağlı olduğunu gösterir.

## 7. Sonuç, Yorum ve Sınırlamalar

**Bulgular (tema bağlamında):**
- Soru 1: ... (çift tepe / commute)
- Soru 2: ... (yaz düşüşü)
- Soru 3: ... (Pareto / yoğun koridorlar)
- Model: baseline'a göre iyileşme ve en belirleyici değişken.

**Sınırlamalar:**
- Yalnızca 2023'ün 4 ayı ve tek ulaşım türü (otobüs) kullanıldı.
- Zamansal ayrım uygulandı; yine de tek aylık test penceresi dardır.
- Hava durumu, etkinlikler gibi dış değişkenler dahil edilmedi.

**Öğrenilenler (vibe coding öz-değerlendirme):**
- Yapay zekânın ürettiği koddan neyi olduğu gibi aldım, neyi değiştirdim?
- Hangi noktada AI'nın çıktısını düzelttim / doğruladım? (örn. veri sızıntısı fark edildi)
- Sürecin en çok zaman alan kısmı neydi?
